# Owner Assets Current Contract Audit

Checks owner-assets endpoint coverage using current contract history, previously successful Ejari creation progress files, or individual success detail JSON files.

Current contract statuses are `Active`, `Pending`, and termination-request statuses. Cancelled and terminated history is intentionally ignored.

Audit sources:

1. `current-contracts`: current contract-history properties only.
2. `latest-successes`: most recent Ejari creation `progress.json` with `ejari_creation_success` records.
3. `uploaded-success-file`: one or more Ejari creation `progress.json` or `success_*.json` files; multiple valid files are merged into a unique success reference before checking.

Expected endpoint logic:

- current owner contract role -> `owner-assets/owned/{2,3}` or `owner-assets/leased/{2,3}`
- current tenant contract role -> `owner-assets/rented/{2,3}`
- previous success reference with no current contract -> property should still exist in `owned` or `leased`
- previous success reference with current tenant role -> both owner side and rented side are checked

Outputs are written to `runs/current_contract_owner_assets_audit_<timestamp>/` or `runs/success_reference_owner_assets_audit_<timestamp>/`.


## Configuration

Loads local `.env` or Colab userdata, defines Emirates IDs, and optionally allows a property-ID filter.

Leave `TARGET_PROPERTY_IDS_BY_EID = {}` to audit every current contract-history property for each configured Emirates ID. Fill it only when you want to restrict the audit to a known list.


In [1]:
# @title Configuration and environment
import base64
import csv
import json
import os
import re
from datetime import datetime
from pathlib import Path

import requests

from notebook_config import (
    DDA_OWNER_EMIRATES_IDS,
    OWNER_EMIRATES_IDS,
    PERSONAL_OWNER_EMIRATES_ID,
)
from notebook_operator_utils import (
    ask_yes_no,
    choose_option,
    select_emirates_ids_for_section,
    select_file_paths,
)
from notebook_progress_utils import (
    count_progress_success_records,
    find_latest_ejari_creation_success_progress,
    iter_progress_success_records,
    load_progress_file,
    resolve_uploaded_success_progress_paths,
)

try:
    from google.colab import userdata
except Exception:
    userdata = None


PROPERTY_TYPE_IDS = (2, 3)
OWNER_ASSET_LIST_TYPES = ("owned", "leased", "rented")
CURRENT_CONTRACT_STATUSES = ("active", "pending", "termination request")
REQUEST_TIMEOUT_SECONDS = 90
CURRENT_BEARER_TOKEN = None

# Optional filter. Example:
# TARGET_PROPERTY_IDS_BY_EID = {"784195279540512": {172672, 17811451, 31137749}}
TARGET_PROPERTY_IDS_BY_EID = {}


def load_local_env(path=".env"):
    env_path = Path(path)
    if not env_path.exists():
        return
    with env_path.open("r", encoding="utf-8") as f:
        for raw_line in f:
            line = raw_line.strip()
            if not line or line.startswith("#") or "=" not in line:
                continue
            key, value = line.split("=", 1)
            os.environ.setdefault(key.strip(), value.strip().strip('"').strip("'"))


load_local_env()


def get_secret(name, default=None):
    if userdata is not None:
        value = userdata.get(name)
        if value is not None:
            return value
    return os.environ.get(name, default)


def require_secret(name):
    value = get_secret(name)
    if not value:
        raise RuntimeError(f"Missing required secret/env var: {name}")
    return value


BASIC_AUTH = require_secret("BASIC_AUTH")
CONSUMER_ID = require_secret("CONSUMER_ID")
DLD_BASE_URL = require_secret("DLD_BASE_URL").rstrip("/")
DLD_PROXY_URL = (get_secret("DLD_PROXY_URL", "") or "").rstrip("/")
IDS_BASE_URL = require_secret("IDS_BASE_URL")
REQUEST_TIMEOUT_SECONDS = int(get_secret("REQUEST_TIMEOUT_SECONDS", REQUEST_TIMEOUT_SECONDS) or REQUEST_TIMEOUT_SECONDS)

print("Configured Emirates IDs:")
for eid in OWNER_EMIRATES_IDS:
    print("-", eid)

if TARGET_PROPERTY_IDS_BY_EID:
    print("Target property filter is enabled:")
    print(json.dumps({k: sorted(str(vv) for vv in v) for k, v in TARGET_PROPERTY_IDS_BY_EID.items()}, indent=2))
else:
    print("Target property filter is disabled; auditing all current contract-history properties.")


Configured Emirates IDs:
- 784199515347708
- 784199668638416
- 784197265140323
- 784195279540512
- 784198721116758
Target property filter is disabled; auditing all current contract-history properties.


## Authentication And API Safety


In [2]:
# @title Authentication and API safety helpers
class ApiRequestError(Exception):
    def __init__(self, message, *, url=None, status_code=None, response=None):
        super().__init__(message)
        self.url = url
        self.status_code = status_code
        self.response = response


def decode_json_like(value):
    if isinstance(value, str):
        text = value.strip()
        if text.startswith("{") or text.startswith("["):
            try:
                return decode_json_like(json.loads(text))
            except Exception:
                return value
        return value
    if isinstance(value, dict):
        return {key: decode_json_like(inner_value) for key, inner_value in value.items()}
    if isinstance(value, list):
        return [decode_json_like(item) for item in value]
    return value


def response_payload(response):
    try:
        return decode_json_like(response.json())
    except Exception:
        return decode_json_like(response.text)


def compact_payload(value, max_chars=1500):
    if isinstance(value, str):
        text = value
    else:
        text = json.dumps(value, ensure_ascii=False, default=str)
    return text if len(text) <= max_chars else text[:max_chars] + "..."


def api_errors(payload):
    if not isinstance(payload, dict):
        return []
    errors = []
    response_code = payload.get("ResponseCode") or payload.get("responseCode")
    if response_code is not None:
        normalized_response_code = str(response_code).strip().lower()
        if normalized_response_code not in {"success", "200", "ok"}:
            errors.append(f"ResponseCode={response_code}")

    benign_messages = {"noerrors", "no errors", "no errors found", "no errors found."}
    for key in ("ErrorMessage", "errorMessage", "Message", "message"):
        value = payload.get(key)
        if value and str(value).strip().lower() not in benign_messages:
            errors.append(str(value))

    for error in payload.get("ValidationErrorsList") or payload.get("validationErrorsList") or []:
        if not isinstance(error, dict):
            errors.append(str(error))
            continue
        error_number = error.get("ErrorNumber") or error.get("errorNumber")
        message_set = error.get("ErrorMessageSet") or error.get("errorMessageSet") or {}
        message = error.get("ErrorMessage") or error.get("errorMessage")
        if isinstance(message_set, dict):
            message = message or message_set.get("EnglishName") or message_set.get("englishName") or message_set.get("ArabicName")
        normalized_message = str(message or "").strip().lower()
        if error_number not in (None, 0, "0"):
            errors.append(f"ErrorNumber={error_number}; {message or compact_payload(error)}")
        elif message and normalized_message not in benign_messages:
            errors.append(str(message))
    return errors


def assert_ok_response(response, api_name, url):
    data = response_payload(response)
    if response.status_code >= 400:
        raise ApiRequestError(
            f"{api_name} failed HTTP {response.status_code}: {compact_payload(data)}",
            url=url,
            status_code=response.status_code,
            response=data,
        )
    errors = api_errors(data)
    if errors:
        raise ApiRequestError(f"{api_name} returned API errors: {' | '.join(errors)}", url=url, status_code=response.status_code, response=data)
    return data


def get_bearer_token():
    url = IDS_BASE_URL
    headers = {
        "Authorization": f"Basic {BASIC_AUTH}",
        "Content-Type": "application/x-www-form-urlencoded",
    }
    response = requests.post(url, headers=headers, data={}, timeout=REQUEST_TIMEOUT_SECONDS)
    data = response_payload(response)
    if response.status_code >= 400:
        raise ApiRequestError(
            f"iPaaS bearer token failed HTTP {response.status_code}: {compact_payload(data)}",
            url=url,
            status_code=response.status_code,
            response=data,
        )
    token = data.get("id_token") if isinstance(data, dict) else None
    if not token:
        raise ApiRequestError(f"iPaaS bearer token response has no id_token: {compact_payload(data)}", url=url, response=data)

    global CURRENT_BEARER_TOKEN
    CURRENT_BEARER_TOKEN = token
    return token


def get_active_bearer_token(force_refresh=False):
    global CURRENT_BEARER_TOKEN
    if force_refresh or not CURRENT_BEARER_TOKEN:
        return get_bearer_token()
    return CURRENT_BEARER_TOKEN


def request_with_bearer(method, url, headers=None, retry_on_401=True, **kwargs):
    request_headers = dict(headers or {})
    request_headers["Authorization"] = f"Bearer {get_active_bearer_token()}"
    kwargs.setdefault("timeout", REQUEST_TIMEOUT_SECONDS)
    response = requests.request(method, url, headers=request_headers, **kwargs)

    if response.status_code == 401 and retry_on_401:
        print("iPaaS bearer token expired; refreshing and retrying request once.")
        request_headers["Authorization"] = f"Bearer {get_active_bearer_token(force_refresh=True)}"
        response = requests.request(method, url, headers=request_headers, **kwargs)

    return response


def dld_headers(dld_token, *, accept_text=False):
    headers = {
        "Token": dld_token,
        "consumer-id": CONSUMER_ID,
    }
    if accept_text:
        headers["accept"] = "text/plain"
    return headers


def get_dld_token(emirates_id):
    bearer_token = get_active_bearer_token()
    url = f"{DLD_BASE_URL}/generaltokenservice/1.0.0/auth"
    auth_obj = {
        "Method": "EmiratesId",
        "MethodIdentity": str(emirates_id),
        "MethodPasscode": "",
        "DeviceKey": "MyPC",
        "ApplicationKey": "DubaiNow",
        "Platform": "Mobile",
    }
    encoded = base64.b64encode(json.dumps(auth_obj).encode()).decode()
    headers = {
        "consumer-id": CONSUMER_ID,
        "x-nv-header": encoded,
        "Content-Type": "application/json",
        "Authorization": f"Bearer {bearer_token}",
    }
    response = request_with_bearer("post", url, headers=headers)
    data = assert_ok_response(response, "DLD token", url)
    token = data.get("token") if isinstance(data, dict) else None
    if not token:
        raise ApiRequestError(f"DLD token response has no token: {compact_payload(data)}", url=url, response=data)
    return token


## Normalization And Matching Helpers


In [3]:
# @title Normalization and matching helpers
def display_name(value):
    if isinstance(value, dict):
        return value.get("EnglishName") or value.get("englishName") or value.get("ArabicName") or value.get("arabicName") or value.get("Value") or value.get("Name")
    return value


def nested_get(value, *keys):
    current = value
    for key in keys:
        if isinstance(current, dict):
            current = current.get(key)
        elif isinstance(current, list) and isinstance(key, int) and len(current) > key:
            current = current[key]
        else:
            return None
    return current


def first_present(*values):
    for value in values:
        if value is None:
            continue
        if isinstance(value, str) and not value.strip():
            continue
        return value
    return None


def compact_key(value):
    if value is None:
        return None
    text = str(value).strip()
    return text or None


def normalize_text(value):
    text = str(value or "").strip().lower()
    return re.sub(r"[^a-z0-9]+", "", text)


def normalize_status(value):
    return normalize_text(display_name(value))


def join_unique(values):
    cleaned = []
    for value in values or []:
        if value is None:
            continue
        text = str(value).strip()
        if text and text not in cleaned:
            cleaned.append(text)
    return "; ".join(cleaned)


def contract_status_values(contract):
    if not isinstance(contract, dict):
        return []
    status = contract.get("ContractStatus") or contract.get("Status")
    return [
        display_name(status),
        nested_get(status, "Identity"),
        nested_get(status, "Value"),
        contract.get("ContractStatusName"),
        contract.get("StatusName"),
        contract.get("ContractStatus") if not isinstance(contract.get("ContractStatus"), dict) else None,
        contract.get("Status") if not isinstance(contract.get("Status"), dict) else None,
    ]


def contract_status_text(contract):
    for value in contract_status_values(contract):
        text = compact_key(display_name(value))
        if text:
            return text
    return None


def is_current_contract_status(contract):
    normalized_values = [normalize_status(value) for value in contract_status_values(contract)]
    normalized_values = [value for value in normalized_values if value]
    for value in normalized_values:
        if value in {"active", "pending"}:
            return True
        if "termin" in value and "request" in value:
            return True
    return False


def property_id_from(item):
    if not isinstance(item, dict):
        return None
    return first_present(
        item.get("property_id"),
        item.get("PropertyId"),
        item.get("PropertyID"),
        item.get("PROPERTY_ID"),
        item.get("EjariPropertyId"),
        nested_get(item, "Property", "PropertyId"),
        nested_get(item, "Property", "PropertyID"),
        nested_get(item, "AssociatedProperty", "PropertyId"),
        nested_get(item, "AssociatedProperty", "PropertyID"),
    )


def property_row_value_from_item(item):
    if not isinstance(item, dict):
        return None
    return first_present(
        item.get("property_row_value"),
        item.get("PropertyRowValue"),
        item.get("propertyRowValue"),
        item.get("RowValue"),
        item.get("DataRowValue"),
        nested_get(item, "Property", "RowValue"),
        nested_get(item, "Property", "PropertyRowValue"),
        nested_get(item, "AssociatedProperty", "RowValue"),
        nested_get(item, "AssociatedProperty", "PropertyRowValue"),
    )


def contract_row_value_from_contract(contract):
    if not isinstance(contract, dict):
        return None
    return first_present(
        contract.get("AssociatedContractRowValue"),
        contract.get("ContractRowValue"),
        contract.get("DataRowValue"),
        nested_get(contract, "AssociatedTenancyContract", "RowValue"),
    )


def contract_number_from_contract(contract):
    if not isinstance(contract, dict):
        return None
    return first_present(
        contract.get("ContractNumber"),
        nested_get(contract, "AssociatedTenancyContract", "ContractNumber"),
    )


def property_type_id_from_item(item):
    if not isinstance(item, dict):
        return None
    return first_present(
        item.get("property_type_id"),
        item.get("PropertyTypeId"),
        item.get("PropertyTypeID"),
        item.get("PROPERTY_TYPE_ID"),
        item.get("PropertyType"),
        nested_get(item, "PropertyType", "Identity"),
        nested_get(item, "PropertyType", "Value"),
        nested_get(item, "Property", "PropertyTypeId"),
        nested_get(item, "AssociatedProperty", "PropertyTypeId"),
    )


def property_title_from_item(item):
    if not isinstance(item, dict):
        return None
    return display_name(first_present(
        item.get("Title"),
        item.get("title"),
        item.get("PropertyName"),
        item.get("property_name"),
        item.get("PropertyNumber"),
        item.get("Name"),
        nested_get(item, "Property", "Title"),
        nested_get(item, "Property", "PropertyName"),
        nested_get(item, "Property", "PropertyNumber"),
        nested_get(item, "AssociatedProperty", "Title"),
        nested_get(item, "AssociatedProperty", "PropertyName"),
        nested_get(item, "AssociatedProperty", "PropertyNumber"),
    ))


def row_match_keys(*items, target_property_id=None, target_title=None):
    keys = []
    if target_property_id is not None:
        keys.append(("property_id", str(target_property_id)))
    if target_title:
        keys.append(("property_title", normalize_text(target_title)))
    for item in items:
        if not isinstance(item, dict):
            continue
        property_id = compact_key(property_id_from(item))
        property_row_value = compact_key(property_row_value_from_item(item))
        title = normalize_text(property_title_from_item(item))
        if property_id:
            keys.append(("property_id", property_id))
        if property_row_value:
            keys.append(("property_row_value", property_row_value))
        if title:
            keys.append(("property_title", title))
    deduped = []
    seen = set()
    for bucket, key in keys:
        marker = (bucket, key)
        if key and marker not in seen:
            seen.add(marker)
            deduped.append(marker)
    return deduped


def item_matches(item, keys):
    return bool(set(row_match_keys(item)).intersection(set(keys)))


def role_label(roles):
    role_set = set(roles)
    if role_set == {"owner", "tenant"}:
        return "both owner and tenant"
    if "owner" in role_set:
        return "owner"
    if "tenant" in role_set:
        return "tenant"
    return "not found in contract history"


def yes_no_na(matches, should_check):
    if not should_check:
        return "NA"
    return "Yes" if matches else "No"


def locations(matches):
    return join_unique(f"{match.get('_owner_asset_list_type')}/{match.get('_property_type_id')}" for match in matches)


## DLD API Calls


In [4]:
# @title Contract history and owner-assets API calls
def get_properties_list(dld_token, list_type, property_type_id):
    url = f"{DLD_BASE_URL}/generalservices/1.0.0/owner-assets/{list_type}/{property_type_id}"
    response = request_with_bearer("get", url, headers=dld_headers(dld_token))
    data = assert_ok_response(response, f"owner-assets {list_type}/{property_type_id}", url)
    properties = nested_get(data, "Response", "Properties")
    if not isinstance(properties, list):
        raise ApiRequestError(f"owner-assets {list_type}/{property_type_id} returned no Response.Properties list", url=url, response=data)
    return properties, data, url


def get_contract_history(dld_token, source="actual"):
    if source == "proxy":
        if not DLD_PROXY_URL:
            raise RuntimeError("DLD_PROXY_URL is required when history_source='proxy'")
        url = f"{DLD_PROXY_URL}/ejariservices/properties/getcontracts"
    else:
        url = f"{DLD_BASE_URL}/dxbnwejaricontracts/1.0.0/properties/getcontracts"
    response = request_with_bearer("get", url, headers=dld_headers(dld_token, accept_text=True))
    data = assert_ok_response(response, f"Contract history ({source})", url)
    response_data = data.get("Response") if isinstance(data, dict) else {}
    owner_contracts = response_data.get("OwnerContracts") if isinstance(response_data, dict) else []
    tenant_contracts = response_data.get("TenantContracts") if isinstance(response_data, dict) else []
    owner_contracts = owner_contracts if isinstance(owner_contracts, list) else []
    tenant_contracts = tenant_contracts if isinstance(tenant_contracts, list) else []

    contracts = []
    for contract in owner_contracts:
        if isinstance(contract, dict) and is_current_contract_status(contract):
            item = dict(contract)
            item["_history_role"] = "owner"
            item["_history_group"] = "OwnerContracts"
            contracts.append(item)
    for contract in tenant_contracts:
        if isinstance(contract, dict) and is_current_contract_status(contract):
            item = dict(contract)
            item["_history_role"] = "tenant"
            item["_history_group"] = "TenantContracts"
            contracts.append(item)
    return contracts, data, url


def fetch_owner_assets(dld_token):
    all_assets = []
    counts = {}
    errors = []
    for list_type in OWNER_ASSET_LIST_TYPES:
        for property_type_id in PROPERTY_TYPE_IDS:
            key = f"{list_type}/{property_type_id}"
            try:
                properties, _, url = get_properties_list(dld_token, list_type, property_type_id)
                counts[key] = len(properties)
                for prop in properties:
                    if not isinstance(prop, dict):
                        continue
                    item = dict(prop)
                    item["_owner_asset_list_type"] = list_type
                    item["_property_type_id"] = property_type_id
                    item["_owner_asset_list_url"] = url
                    all_assets.append(item)
            except Exception as exc:
                counts[key] = None
                errors.append({
                    "owner_asset_list": key,
                    "error": str(exc),
                    "url": getattr(exc, "url", None),
                    "status_code": getattr(exc, "status_code", None),
                    "response": compact_payload(getattr(exc, "response", None), 800),
                })
    return all_assets, counts, errors


def find_assets(assets, keys, list_types):
    matches = []
    seen = set()
    for asset in assets:
        if asset.get("_owner_asset_list_type") not in list_types:
            continue
        if not item_matches(asset, keys):
            continue
        marker = (
            asset.get("_owner_asset_list_type"),
            asset.get("_property_type_id"),
            compact_key(property_id_from(asset)),
            compact_key(property_row_value_from_item(asset)),
        )
        if marker in seen:
            continue
        seen.add(marker)
        matches.append(asset)
    return matches


## Comparison Logic


In [5]:
# @title Comparison logic and output rows
def property_filter_allows(emirates_id, contract):
    configured = TARGET_PROPERTY_IDS_BY_EID.get(str(emirates_id)) or TARGET_PROPERTY_IDS_BY_EID.get(emirates_id)
    if not configured:
        return True
    property_id = property_id_from(contract)
    return str(property_id) in {str(value) for value in configured}


def contract_group_key(contract):
    property_id = compact_key(property_id_from(contract))
    property_row_value = compact_key(property_row_value_from_item(contract))
    title = normalize_text(property_title_from_item(contract))
    if property_id:
        return ("property_id", property_id)
    if property_row_value:
        return ("property_row_value", property_row_value)
    if title:
        return ("property_title", title)
    return ("contract_row_value", compact_key(contract_row_value_from_contract(contract)) or str(id(contract)))


def group_current_contracts(contracts):
    groups = {}
    for contract in contracts:
        key = contract_group_key(contract)
        groups.setdefault(key, []).append(contract)
    return groups


def matching_current_contracts_for_success(record, contracts):
    keys = row_match_keys(record)
    matches = []
    seen = set()
    for contract in contracts:
        if item_matches(contract, keys):
            marker = (
                contract.get("_history_role"),
                contract_number_from_contract(contract),
                contract_row_value_from_contract(contract),
                property_id_from(contract),
                property_row_value_from_item(contract),
            )
            if marker in seen:
                continue
            seen.add(marker)
            matches.append(contract)
    return matches


def matching_progress_success_items_for_items(items, success_items):
    keys = row_match_keys(*items)
    matches = []
    seen = set()
    for item in success_items or []:
        record = item.get("record") if isinstance(item, dict) else None
        if not isinstance(record, dict) or not item_matches(record, keys):
            continue
        marker = (
            item.get("emirates_id"),
            item.get("property_key"),
            property_id_from(record),
            property_row_value_from_item(record),
            normalize_text(property_title_from_item(record)),
            item.get("progress_path"),
        )
        if marker in seen:
            continue
        seen.add(marker)
        matches.append(item)
    return matches


def yes_no_not_checked(value, checked):
    if not checked:
        return "not checked"
    return "Yes" if value else "No"


def split_multi_value(value):
    if value is None:
        return []
    parts = []
    for part in str(value).replace(",", ";").split(";"):
        text = part.strip()
        if text and text not in parts:
            parts.append(text)
    return parts


def shell_single_quote(value):
    return "'" + str(value).replace("'", "'\"'\"'") + "'"


def owner_assets_property_detail_url(list_type, property_type_id, property_row_value):
    return f"{DLD_BASE_URL}/generalservices/1.0.0/owner-assets/{list_type}/{property_type_id}/{property_row_value}"


def owner_assets_property_detail_curl(emirates_id, list_type, property_type_id, property_row_value):
    url = owner_assets_property_detail_url(list_type, property_type_id, property_row_value)
    headers = [
        ("Authorization", "Bearer <IPAAS_BEARER_TOKEN>"),
        ("Token", f"<DLD_TOKEN_FOR_EID_{emirates_id}>"),
        ("consumer-id", CONSUMER_ID),
    ]
    line_continuation = " \\" + "\n  "
    curl = f"curl --location --request GET {shell_single_quote(url)}"
    for key, value in headers:
        curl += line_continuation + f"--header {shell_single_quote(f'{key}: {value}')}"
    return curl


def problematic_detail_list_types(row):
    problem = str(row.get("Problem") or "")
    list_types = []
    if "Missing from owned/leased" in problem:
        list_types.extend(["owned", "leased"])
    if "Missing from rented" in problem:
        list_types.append("rented")
    return [list_type for index, list_type in enumerate(list_types) if list_type not in list_types[:index]]


def problematic_detail_property_type_ids(row):
    type_ids = [value for value in split_multi_value(row.get("Property type id(s)")) if value in {"2", "3"}]
    return type_ids or [str(value) for value in PROPERTY_TYPE_IDS]


def problematic_detail_property_row_values(row):
    return split_multi_value(row.get("Property Row Value"))


def build_problematic_property_detail_curls(row):
    curls = []
    for list_type in problematic_detail_list_types(row):
        for property_type_id in problematic_detail_property_type_ids(row):
            for property_row_value in problematic_detail_property_row_values(row):
                curls.append(owner_assets_property_detail_curl(row.get("EID"), list_type, property_type_id, property_row_value))
    return curls


def attach_problematic_property_detail_curls(row):
    if row.get("Problem"):
        row["Problematic property detail curl(s)"] = "\n\n".join(build_problematic_property_detail_curls(row))
    else:
        row["Problematic property detail curl(s)"] = ""
    return row


def problem_row_display_title(row):
    return (
        f"EID {row.get('EID') or ''} | "
        f"Property {row.get('Property Id') or ''} | "
        f"{row.get('Property Name') or ''} | "
        f"Row {row.get('Property Row Value') or ''}"
    )


def display_problem_rows_collapsed(problem_rows):
    if not problem_rows:
        print("No problematic properties found.")
        return
    try:
        import html
        from IPython.display import HTML, display
    except Exception:
        print(json.dumps(problem_rows, ensure_ascii=False, indent=2, default=str))
        return

    blocks = [
        "<div style='font-family:system-ui,Segoe UI,Arial,sans-serif;'>",
        f"<div style='font-weight:700;margin:8px 0;'>Problematic properties: {len(problem_rows)}</div>",
    ]
    for index, row in enumerate(problem_rows, start=1):
        row = attach_problematic_property_detail_curls(dict(row))
        display_row = dict(row)
        display_row.pop("Problematic property detail curl(s)", None)
        title = html.escape(problem_row_display_title(row))
        row_json = html.escape(json.dumps(display_row, ensure_ascii=False, indent=2, default=str))
        curls = html.escape(row.get("Problematic property detail curl(s)") or "No property-detail curl could be generated. Check property row value and property type id.")
        problem = html.escape(row.get("Problem") or "")
        blocks.append(
            f"""
            <details style="border:1px solid #d0d7de;border-radius:6px;margin:8px 0;background:#fff;">
              <summary style="cursor:pointer;padding:10px 12px;font-weight:600;background:#f6f8fa;border-radius:6px;">
                {index}. {title}
              </summary>
              <div style="padding:10px 12px;">
                <div style="margin-bottom:8px;"><strong>Problem:</strong> {problem}</div>
                <details style="border:1px solid #e5e7eb;border-radius:6px;margin:8px 0;">
                  <summary style="cursor:pointer;padding:8px 10px;background:#f9fafb;font-weight:600;">Row JSON</summary>
                  <pre style="white-space:pre-wrap;background:#0b1020;color:#f8fafc;padding:12px;margin:0;border-radius:0 0 6px 6px;overflow:auto;">{row_json}</pre>
                </details>
                <details style="border:1px solid #e5e7eb;border-radius:6px;margin:8px 0;">
                  <summary style="cursor:pointer;padding:8px 10px;background:#f9fafb;font-weight:600;">Problematic property-detail curl(s)</summary>
                  <pre style="white-space:pre-wrap;background:#111827;color:#e5e7eb;padding:12px;margin:0;border-radius:0 0 6px 6px;overflow:auto;">{curls}</pre>
                </details>
              </div>
            </details>
            """
        )
    blocks.append("</div>")
    display(HTML("\n".join(blocks)))


def build_result_row(
    emirates_id,
    contracts,
    owner_assets,
    *,
    source="current_contract_history",
    progress_success_items=None,
    progress_reference_checked=False,
):
    roles = sorted({contract.get("_history_role") for contract in contracts if contract.get("_history_role")})
    keys = row_match_keys(*contracts)
    owner_matches = find_assets(owner_assets, keys, {"owned", "leased"})
    rented_matches = find_assets(owner_assets, keys, {"rented"})
    matching_progress_items = matching_progress_success_items_for_items(contracts, progress_success_items)
    should_check_owner = "owner" in roles
    should_check_rented = "tenant" in roles
    problem_reasons = []
    if should_check_owner and not owner_matches:
        problem_reasons.append("Missing from owned/leased")
    if should_check_rented and not rented_matches:
        problem_reasons.append("Missing from rented")

    return {
        "Source": source,
        "EID": emirates_id,
        "Property Name": join_unique([property_title_from_item(contract) for contract in contracts] + [property_title_from_item(item.get("record")) for item in matching_progress_items] + [property_title_from_item(asset) for asset in owner_matches + rented_matches]),
        "Property Row Value": join_unique([property_row_value_from_item(contract) for contract in contracts] + [property_row_value_from_item(item.get("record")) for item in matching_progress_items] + [property_row_value_from_item(asset) for asset in owner_matches + rented_matches]),
        "Property Id": first_present(*[property_id_from(contract) for contract in contracts], *[property_id_from(item.get("record")) for item in matching_progress_items], *[property_id_from(asset) for asset in owner_matches + rented_matches]),
        "Contract": role_label(roles),
        "Property present in Owned/Leased endpoint": yes_no_na(owner_matches, should_check_owner),
        "Property present in Rented": yes_no_na(rented_matches, should_check_rented),
        "Problem": "; ".join(problem_reasons),
        "Present in current contract history": "Yes",
        "Present in progress/success reference": yes_no_not_checked(matching_progress_items, progress_reference_checked),
        "Owned/Leased match endpoint(s)": locations(owner_matches),
        "Rented match endpoint(s)": locations(rented_matches),
        "Contract status(es)": join_unique(contract_status_text(contract) for contract in contracts),
        "Contract number(s)": join_unique(contract_number_from_contract(contract) for contract in contracts),
        "Contract row value(s)": join_unique(contract_row_value_from_contract(contract) for contract in contracts),
        "Contract history group(s)": join_unique(contract.get("_history_group") for contract in contracts),
        "Property type id(s)": join_unique([property_type_id_from_item(contract) for contract in contracts] + [property_type_id_from_item(item.get("record")) for item in matching_progress_items] + [asset.get("_property_type_id") for asset in owner_matches + rented_matches]),
        "Progress success file(s)": join_unique(item.get("progress_path") for item in matching_progress_items),
        "Progress success property key(s)": join_unique(item.get("property_key") for item in matching_progress_items),
    }


def build_success_reference_row(emirates_id, success_items, current_contracts, owner_assets, *, contract_history_checked=True):
    records = [item["record"] for item in success_items]
    matching_contracts = []
    seen_contracts = set()
    if contract_history_checked:
        for record in records:
            for contract in matching_current_contracts_for_success(record, current_contracts):
                marker = (
                    contract.get("_history_role"),
                    contract_number_from_contract(contract),
                    contract_row_value_from_contract(contract),
                    property_id_from(contract),
                    property_row_value_from_item(contract),
                )
                if marker in seen_contracts:
                    continue
                seen_contracts.add(marker)
                matching_contracts.append(contract)

    roles = sorted({contract.get("_history_role") for contract in matching_contracts if contract.get("_history_role")})
    keys = row_match_keys(*records, *matching_contracts)
    owner_matches = find_assets(owner_assets, keys, {"owned", "leased"})
    rented_matches = find_assets(owner_assets, keys, {"rented"})

    should_check_owner = True
    should_check_rented = contract_history_checked and "tenant" in roles
    problem_reasons = []
    if should_check_owner and not owner_matches:
        problem_reasons.append("Missing from owned/leased")
    if should_check_rented and not rented_matches:
        problem_reasons.append("Missing from rented")

    return {
        "Source": "progress_success_reference",
        "EID": emirates_id,
        "Property Name": join_unique([property_title_from_item(record) for record in records] + [property_title_from_item(contract) for contract in matching_contracts] + [property_title_from_item(asset) for asset in owner_matches + rented_matches]),
        "Property Row Value": join_unique([property_row_value_from_item(record) for record in records] + [property_row_value_from_item(contract) for contract in matching_contracts] + [property_row_value_from_item(asset) for asset in owner_matches + rented_matches]),
        "Property Id": first_present(*[property_id_from(record) for record in records], *[property_id_from(contract) for contract in matching_contracts], *[property_id_from(asset) for asset in owner_matches + rented_matches]),
        "Contract": role_label(roles) if contract_history_checked else "not checked",
        "Property present in Owned/Leased endpoint": yes_no_na(owner_matches, should_check_owner),
        "Property present in Rented": yes_no_na(rented_matches, should_check_rented),
        "Problem": "; ".join(problem_reasons),
        "Present in current contract history": yes_no_not_checked(matching_contracts, contract_history_checked),
        "Present in progress/success reference": "Yes",
        "Owned/Leased match endpoint(s)": locations(owner_matches),
        "Rented match endpoint(s)": locations(rented_matches),
        "Contract status(es)": join_unique(contract_status_text(contract) for contract in matching_contracts),
        "Contract number(s)": join_unique(contract_number_from_contract(contract) for contract in matching_contracts),
        "Contract row value(s)": join_unique(contract_row_value_from_contract(contract) for contract in matching_contracts),
        "Contract history group(s)": join_unique(contract.get("_history_group") for contract in matching_contracts),
        "Property type id(s)": join_unique([property_type_id_from_item(record) for record in records] + [property_type_id_from_item(contract) for contract in matching_contracts] + [asset.get("_property_type_id") for asset in owner_matches + rented_matches]),
        "Progress success file(s)": join_unique(item.get("progress_path") for item in success_items),
        "Progress success property key(s)": join_unique(item.get("property_key") for item in success_items),
    }


RESULT_COLUMNS = [
    "Source",
    "EID",
    "Property Name",
    "Property Row Value",
    "Property Id",
    "Contract",
    "Property present in Owned/Leased endpoint",
    "Property present in Rented",
    "Problem",
    "Problematic property detail curl(s)",
    "Present in current contract history",
    "Present in progress/success reference",
    "Owned/Leased match endpoint(s)",
    "Rented match endpoint(s)",
    "Contract status(es)",
    "Contract number(s)",
    "Contract row value(s)",
    "Contract history group(s)",
    "Property type id(s)",
    "Progress success file(s)",
    "Progress success property key(s)",
]

SUMMARY_COLUMNS = [
    "source",
    "emirates_id",
    "contract_history_cross_check_enabled",
    "progress_success_reference_enabled",
    "contract_history_current_rows",
    "contract_history_current_properties",
    "progress_success_records",
    "progress_success_properties",
    "problematic_properties",
    "problematic_properties_present_in_contract_history",
    "problematic_properties_present_in_progress_success",
    "owned_2_count",
    "owned_3_count",
    "leased_2_count",
    "leased_3_count",
    "rented_2_count",
    "rented_3_count",
    "owner_asset_fetch_error_count",
]

ERROR_COLUMNS = ["emirates_id", "stage", "owner_asset_list", "error", "url", "status_code", "response"]


def write_csv(path, rows, columns):
    with Path(path).open("w", newline="", encoding="utf-8-sig") as f:
        writer = csv.DictWriter(f, fieldnames=columns, extrasaction="ignore")
        writer.writeheader()
        writer.writerows(rows)


def write_json(path, rows):
    Path(path).write_text(json.dumps(rows, ensure_ascii=False, indent=2, default=str), encoding="utf-8")


## Run Audit


In [6]:
# @title Run owner-assets audit
def save_owner_assets_audit_outputs(output_dir, all_rows, problem_rows, summary_rows, error_rows, *, all_filename, problem_filename):
    write_csv(output_dir / all_filename, all_rows, RESULT_COLUMNS)
    write_csv(output_dir / problem_filename, problem_rows, RESULT_COLUMNS)
    write_csv(output_dir / "summary.csv", summary_rows, SUMMARY_COLUMNS)
    write_csv(output_dir / "errors.csv", error_rows, ERROR_COLUMNS)
    write_json(output_dir / all_filename.replace(".csv", ".json"), all_rows)
    write_json(output_dir / problem_filename.replace(".csv", ".json"), problem_rows)
    write_json(output_dir / "summary.json", summary_rows)
    write_json(output_dir / "errors.json", error_rows)


def group_progress_success_records(success_items):
    groups = {}
    for item in success_items:
        record = item["record"]
        key = progress_success_group_key(item["emirates_id"], item.get("property_key"), record)
        groups.setdefault(key, []).append(item)
    return groups


def progress_success_group_key(emirates_id, property_key, record):
    property_row_value = property_row_value_from_item(record) or property_key
    property_id = property_id_from(record)
    title = normalize_text(property_title_from_item(record))
    if property_row_value:
        return str(emirates_id), "property_row_value", str(property_row_value)
    if property_id:
        return str(emirates_id), "property_id", str(property_id)
    if title:
        return str(emirates_id), "property_title", title
    return str(emirates_id), "property_key", str(property_key)


def success_items_by_owner_from_progress(progress_path):
    if not progress_path:
        return {}, 0, 0
    progress_path = Path(progress_path)
    progress = load_progress_file(progress_path)
    success_items = list(iter_progress_success_records(progress, progress_path))
    grouped_successes = group_progress_success_records(success_items)
    by_owner = {}
    for group_items in grouped_successes.values():
        by_owner.setdefault(group_items[0]["emirates_id"], []).extend(group_items)
    return by_owner, len(success_items), len(grouped_successes)


def count_problematic_present(rows, column_name):
    return sum(1 for row in rows if row.get("Problem") and row.get(column_name) == "Yes")


def run_current_contract_owner_assets_audit(owner_emirates_ids=OWNER_EMIRATES_IDS, history_source="actual", progress_reference_path=None):
    output_dir = Path("runs") / f"current_contract_owner_assets_audit_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
    output_dir.mkdir(parents=True, exist_ok=True)

    progress_reference_checked = progress_reference_path is not None
    progress_items_by_owner = {}
    if progress_reference_checked:
        progress_items_by_owner, total_progress_records, total_progress_properties = success_items_by_owner_from_progress(progress_reference_path)
        print(f"Using progress/success reference: {progress_reference_path}")
        print(f"Progress/success reference records: {total_progress_records}; properties: {total_progress_properties}")

    all_rows = []
    problem_rows = []
    summary_rows = []
    error_rows = []

    selected_emirates_ids = select_emirates_ids_for_section("owner-assets current-contract audit", owner_emirates_ids, default_all=True)
    for emirates_id in selected_emirates_ids:
        print(f"\nProcessing Emirates ID {emirates_id}")
        try:
            global CURRENT_BEARER_TOKEN
            CURRENT_BEARER_TOKEN = None
            dld_token = get_dld_token(emirates_id)

            owner_assets, owner_asset_counts, owner_asset_errors = fetch_owner_assets(dld_token)
            for error in owner_asset_errors:
                error_rows.append({"emirates_id": emirates_id, "stage": "owner_assets_fetch", **error})

            contracts, _, contract_history_url = get_contract_history(dld_token, source=history_source)
            contracts = [contract for contract in contracts if property_filter_allows(emirates_id, contract)]
            grouped_contracts = group_current_contracts(contracts)
            progress_items = progress_items_by_owner.get(str(emirates_id), [])
            progress_groups = group_progress_success_records(progress_items) if progress_reference_checked else {}
            emitted_progress_group_keys = set()

            eid_rows = []
            eid_problem_rows = []
            for contract_group in grouped_contracts.values():
                if progress_reference_checked:
                    matching_progress_items = matching_progress_success_items_for_items(contract_group, progress_items)
                    for item in matching_progress_items:
                        emitted_progress_group_keys.add(progress_success_group_key(item["emirates_id"], item.get("property_key"), item["record"]))
                row = build_result_row(
                    emirates_id,
                    contract_group,
                    owner_assets,
                    progress_success_items=progress_items,
                    progress_reference_checked=progress_reference_checked,
                )
                eid_rows.append(row)
                if row["Problem"]:
                    attach_problematic_property_detail_curls(row)
                    eid_problem_rows.append(row)

            if progress_reference_checked:
                for progress_group_key, success_group in progress_groups.items():
                    if progress_group_key in emitted_progress_group_keys:
                        continue
                    row = build_success_reference_row(
                        emirates_id,
                        success_group,
                        contracts,
                        owner_assets,
                        contract_history_checked=True,
                    )
                    row["Source"] = "progress_success_reference_without_current_contract_group"
                    eid_rows.append(row)
                    if row["Problem"]:
                        attach_problematic_property_detail_curls(row)
                        eid_problem_rows.append(row)

            problem_present_in_contract_history = count_problematic_present(eid_problem_rows, "Present in current contract history")
            problem_present_in_progress = count_problematic_present(eid_problem_rows, "Present in progress/success reference")

            all_rows.extend(eid_rows)
            problem_rows.extend(eid_problem_rows)
            summary_rows.append({
                "source": "current_contract_history",
                "emirates_id": emirates_id,
                "contract_history_cross_check_enabled": True,
                "progress_success_reference_enabled": progress_reference_checked,
                "contract_history_current_rows": len(contracts),
                "contract_history_current_properties": len(grouped_contracts),
                "progress_success_records": len(progress_items) if progress_reference_checked else 0,
                "progress_success_properties": len(progress_groups) if progress_reference_checked else 0,
                "problematic_properties": len(eid_problem_rows),
                "problematic_properties_present_in_contract_history": problem_present_in_contract_history,
                "problematic_properties_present_in_progress_success": problem_present_in_progress,
                "owned_2_count": owner_asset_counts.get("owned/2"),
                "owned_3_count": owner_asset_counts.get("owned/3"),
                "leased_2_count": owner_asset_counts.get("leased/2"),
                "leased_3_count": owner_asset_counts.get("leased/3"),
                "rented_2_count": owner_asset_counts.get("rented/2"),
                "rented_3_count": owner_asset_counts.get("rented/3"),
                "owner_asset_fetch_error_count": len(owner_asset_errors),
                "contract_history_url": contract_history_url,
                "progress_success_file": str(progress_reference_path or ""),
            })

            print(f"Current contract-history rows: {len(contracts)}")
            print(f"Current contract-history properties: {len(grouped_contracts)}")
            if progress_reference_checked:
                print(f"Progress/success reference properties: {len(progress_groups)}")
                print(f"Output comparison properties: {len(eid_rows)}")
            print(f"Problematic properties: {len(eid_problem_rows)}")
            print(f"Problematic properties present in contract history: {problem_present_in_contract_history}")
            if progress_reference_checked:
                print(f"Problematic properties present in progress/success reference: {problem_present_in_progress}")

        except Exception as exc:
            print(f"ERROR processing Emirates ID {emirates_id}: {exc}")
            error_rows.append({
                "emirates_id": emirates_id,
                "stage": "emirates_id_processing",
                "owner_asset_list": None,
                "error": str(exc),
                "url": getattr(exc, "url", None),
                "status_code": getattr(exc, "status_code", None),
                "response": compact_payload(getattr(exc, "response", None), 800),
            })
        finally:
            save_owner_assets_audit_outputs(
                output_dir,
                all_rows,
                problem_rows,
                summary_rows,
                error_rows,
                all_filename="all_current_contract_owner_assets_comparison.csv",
                problem_filename="problematic_current_contract_properties.csv",
            )
            print("Saved current output files.")

    print("\nDone.")
    print("Output directory:", output_dir)
    print("Total output comparison properties:", len(all_rows))
    print("Total problematic properties:", len(problem_rows))
    print("Total problematic properties present in contract history:", count_problematic_present(problem_rows, "Present in current contract history"))
    if progress_reference_checked:
        print("Total problematic properties present in progress/success reference:", count_problematic_present(problem_rows, "Present in progress/success reference"))

    try:
        import pandas as pd
        from IPython.display import display
        display_problem_rows_collapsed(problem_rows)
        display(pd.DataFrame(summary_rows, columns=SUMMARY_COLUMNS))
    except Exception:
        print(json.dumps(problem_rows, ensure_ascii=False, indent=2, default=str))

    return all_rows, problem_rows, summary_rows, error_rows, output_dir


def run_progress_success_owner_assets_audit(progress_path, history_source="actual", cross_check_contract_history=True):
    progress_path = Path(progress_path)
    progress = load_progress_file(progress_path)
    success_items = list(iter_progress_success_records(progress, progress_path))
    grouped_successes = group_progress_success_records(success_items)
    output_dir = Path("runs") / f"success_reference_owner_assets_audit_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
    output_dir.mkdir(parents=True, exist_ok=True)

    all_rows = []
    problem_rows = []
    summary_rows = []
    error_rows = []

    by_owner = {}
    for group_items in grouped_successes.values():
        by_owner.setdefault(group_items[0]["emirates_id"], []).append(group_items)

    selected_emirates_ids = select_emirates_ids_for_section("owner-assets progress/success reference audit", sorted(by_owner), default_all=True)
    for emirates_id in selected_emirates_ids:
        success_groups = by_owner[emirates_id]
        print(f"\nProcessing progress/success references for Emirates ID {emirates_id}")
        try:
            global CURRENT_BEARER_TOKEN
            CURRENT_BEARER_TOKEN = None
            dld_token = get_dld_token(emirates_id)

            owner_assets, owner_asset_counts, owner_asset_errors = fetch_owner_assets(dld_token)
            for error in owner_asset_errors:
                error_rows.append({"emirates_id": emirates_id, "stage": "owner_assets_fetch", **error})

            if cross_check_contract_history:
                contracts, _, contract_history_url = get_contract_history(dld_token, source=history_source)
            else:
                contracts = []
                contract_history_url = ""

            eid_rows = []
            eid_problem_rows = []
            for success_group in success_groups:
                row = build_success_reference_row(
                    emirates_id,
                    success_group,
                    contracts,
                    owner_assets,
                    contract_history_checked=cross_check_contract_history,
                )
                eid_rows.append(row)
                if row["Problem"]:
                    attach_problematic_property_detail_curls(row)
                    eid_problem_rows.append(row)

            problem_present_in_contract_history = count_problematic_present(eid_problem_rows, "Present in current contract history")
            problem_present_in_progress = count_problematic_present(eid_problem_rows, "Present in progress/success reference")

            all_rows.extend(eid_rows)
            problem_rows.extend(eid_problem_rows)
            summary_rows.append({
                "source": "progress_success_reference",
                "emirates_id": emirates_id,
                "contract_history_cross_check_enabled": cross_check_contract_history,
                "progress_success_reference_enabled": True,
                "contract_history_current_rows": len(contracts),
                "contract_history_current_properties": len(group_current_contracts(contracts)) if cross_check_contract_history else 0,
                "progress_success_records": sum(len(group) for group in success_groups),
                "progress_success_properties": len(eid_rows),
                "problematic_properties": len(eid_problem_rows),
                "problematic_properties_present_in_contract_history": problem_present_in_contract_history,
                "problematic_properties_present_in_progress_success": problem_present_in_progress,
                "owned_2_count": owner_asset_counts.get("owned/2"),
                "owned_3_count": owner_asset_counts.get("owned/3"),
                "leased_2_count": owner_asset_counts.get("leased/2"),
                "leased_3_count": owner_asset_counts.get("leased/3"),
                "rented_2_count": owner_asset_counts.get("rented/2"),
                "rented_3_count": owner_asset_counts.get("rented/3"),
                "owner_asset_fetch_error_count": len(owner_asset_errors),
                "contract_history_url": contract_history_url,
                "progress_success_file": str(progress_path),
            })

            print(f"Progress/success properties: {len(eid_rows)}")
            if cross_check_contract_history:
                print(f"Current contract-history rows: {len(contracts)}")
            else:
                print("Current contract-history cross-check: not selected")
            print(f"Problematic properties: {len(eid_problem_rows)}")
            print(f"Problematic properties present in contract history: {problem_present_in_contract_history}")

        except Exception as exc:
            print(f"ERROR processing Emirates ID {emirates_id}: {exc}")
            error_rows.append({
                "emirates_id": emirates_id,
                "stage": "emirates_id_processing",
                "owner_asset_list": None,
                "error": str(exc),
                "url": getattr(exc, "url", None),
                "status_code": getattr(exc, "status_code", None),
                "response": compact_payload(getattr(exc, "response", None), 800),
            })
        finally:
            save_owner_assets_audit_outputs(
                output_dir,
                all_rows,
                problem_rows,
                summary_rows,
                error_rows,
                all_filename="all_success_reference_owner_assets_comparison.csv",
                problem_filename="problematic_success_reference_properties.csv",
            )
            print("Saved current output files.")

    print("\nDone.")
    print("Output directory:", output_dir)
    print("Total progress/success properties:", len(all_rows))
    print("Total problematic properties:", len(problem_rows))
    print("Total problematic properties present in contract history:", count_problematic_present(problem_rows, "Present in current contract history"))

    try:
        import pandas as pd
        from IPython.display import display
        display_problem_rows_collapsed(problem_rows)
        display(pd.DataFrame(summary_rows, columns=SUMMARY_COLUMNS))
    except Exception:
        print(json.dumps(problem_rows, ensure_ascii=False, indent=2, default=str))

    return all_rows, problem_rows, summary_rows, error_rows, output_dir


source_aliases = {
    "1": "current-contracts",
    "current": "current-contracts",
    "current-contract": "current-contracts",
    "contracts": "current-contracts",
    "2": "latest-successes",
    "latest": "latest-successes",
    "latest-success": "latest-successes",
    "recent": "latest-successes",
    "recent-successes": "latest-successes",
    "3": "uploaded-success-file",
    "upload": "uploaded-success-file",
    "uploaded": "uploaded-success-file",
    "uploads": "uploaded-success-file",
    "file": "uploaded-success-file",
    "files": "uploaded-success-file",
    "success-file": "uploaded-success-file",
    "success-files": "uploaded-success-file",
    "multiple": "uploaded-success-file",
    "multiple-files": "uploaded-success-file",
    "no": "no",
    "n": "no",
}


def choose_contract_history_source(prompt="Contract history API source?"):
    return choose_option(
        prompt,
        [
            ("actual", "Actual API"),
            ("proxy", "Proxy API"),
        ],
        default="actual",
        aliases={"1": "actual", "2": "proxy"},
        title="Contract history source",
    )


def choose_uploaded_progress_success_reference(merge_output_prefix):
    print("Select one or more Ejari creation progress.json or success_*.json files.")
    selected_progress_paths = select_file_paths(
        "Select one or more Ejari creation progress.json or success_*.json files",
        default_dir=".",
        multiple=True,
        title="Select progress/success JSON file(s)",
    )
    progress_path, success_count, merge_dir, valid_files, invalid_files = resolve_uploaded_success_progress_paths(
        selected_progress_paths,
        confirm_callback=ask_yes_no,
        merge_output_prefix=merge_output_prefix,
    )
    if merge_dir:
        print(f"Using merged progress/success reference file: {progress_path} ({success_count} successes)")
    else:
        print(f"Using progress/success reference file: {progress_path} ({success_count} successes)")
    return progress_path, success_count


def choose_progress_success_reference_for_current_contracts():
    reference_source = choose_option(
        "Progress/success reference source?",
        [
            ("latest-successes", "Most recent progress.json successes"),
            ("uploaded-success-file", "Select progress.json or success_*.json file(s)"),
        ],
        default="latest-successes",
        aliases=source_aliases,
        title="Progress/success reference source",
    )
    if reference_source == "latest-successes":
        progress_path, success_count = find_latest_ejari_creation_success_progress()
        print(f"Using latest Ejari creation success progress file: {progress_path} ({success_count} successes)")
        return progress_path
    progress_path, _ = choose_uploaded_progress_success_reference("owner_assets_current_contract_progress_success_reference_merge")
    return progress_path


audit_source = choose_option(
    "Owner-assets audit source?",
    [
        ("current-contracts", "Current contract-history audit"),
        ("latest-successes", "Most recent progress.json successes"),
        ("uploaded-success-file", "Select progress/success JSON file(s)"),
        ("no", "Skip audit"),
    ],
    default="current-contracts",
    aliases=source_aliases,
    title="Owner-assets audit source",
)

if audit_source == "current-contracts":
    history_source = choose_contract_history_source("Contract history API source?")
    progress_reference_path = None
    if ask_yes_no("Also compare current contract-history properties with progress/success JSON reference?", default=False):
        progress_reference_path = choose_progress_success_reference_for_current_contracts()
    rows, problematic_rows, summaries, errors, output_dir = run_current_contract_owner_assets_audit(
        OWNER_EMIRATES_IDS,
        history_source=history_source,
        progress_reference_path=progress_reference_path,
    )

elif audit_source == "latest-successes":
    progress_path, success_count = find_latest_ejari_creation_success_progress()
    print(f"Using latest Ejari creation success progress file: {progress_path} ({success_count} successes)")
    cross_check_contract_history = ask_yes_no("Cross-verify these progress/success references with current contract history?", default=True)
    history_source = choose_contract_history_source("Contract history API source?") if cross_check_contract_history else "actual"
    rows, problematic_rows, summaries, errors, output_dir = run_progress_success_owner_assets_audit(
        progress_path,
        history_source=history_source,
        cross_check_contract_history=cross_check_contract_history,
    )

elif audit_source == "uploaded-success-file":
    progress_path, success_count = choose_uploaded_progress_success_reference("owner_assets_uploaded_success_progress_merge")
    cross_check_contract_history = ask_yes_no("Cross-verify these progress/success references with current contract history?", default=True)
    history_source = choose_contract_history_source("Contract history API source?") if cross_check_contract_history else "actual"
    rows, problematic_rows, summaries, errors, output_dir = run_progress_success_owner_assets_audit(
        progress_path,
        history_source=history_source,
        cross_check_contract_history=cross_check_contract_history,
    )

elif audit_source == "no":
    print("Skipped owner-assets audit.")

else:
    raise ValueError("Invalid audit source. Choose current-contracts, latest-successes, uploaded-success-file, or no.")


Using latest Ejari creation success progress file: runs\ejari_creation_merged_18_successes_20260608_110638\progress.json (18 successes)

Processing progress/success references for Emirates ID 784195279540512
ERROR processing Emirates ID 784195279540512: Contract history (actual) failed HTTP 408: {"Exception": "API Gateway encountered an error. Error Message:  Downtime exception: Read timed out. Request Details: Service - SDG-DLD-DXBNWEjariContracts, Operation - /properties/getcontracts, Invocation Time:10:39:17 AM, Date:Jun 10, 2026,  Client IP - 91.72.101.186, User - Default and Application:SDG-DubaiNow-Microservices"}
Saved current output files.

Processing progress/success references for Emirates ID 784197265140323
Progress/success properties: 3
Current contract-history rows: 48
Problematic properties: 0
Problematic properties present in contract history: 0
Saved current output files.

Processing progress/success references for Emirates ID 784199515347708
Progress/success properties